# SOM (Set-of-Mark) Preprocessing

Process images with OmniParser (YOLO + OCR) to detect UI elements and annotate with numbered marks.

In [ ]:
# Install SOM dependencies
%pip install -q ultralytics easyocr opencv-python-headless

In [ ]:
import os
import sys
import numpy as np
import torch
import easyocr
from PIL import Image, ImageDraw
from IPython.display import display

# PyTorch 2.10+ compatibility patch
_original_sum = torch.Tensor.sum
def _patched_sum(self, *args, **kwargs):
    if self.dtype == torch.bool:
        return _original_sum(self.to(torch.int64), *args, **kwargs)
    return _original_sum(self, *args, **kwargs)
torch.Tensor.sum = _patched_sum

# Find project root
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, 'magma', 'modeling_magma.py')):
        break
    project_root = os.path.dirname(project_root)
print(f"Project root: {project_root}")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Try to load SOM utilities
try:
    from agents.ui_agent.util.som import MarkHelper, plot_boxes_with_marks
    som_available = True
    print("SOM utilities loaded")
except ImportError:
    som_available = False
    print("SOM utilities not found - will use basic annotation")

In [ ]:
def process_som(image, ocr_reader=None):
    """Run SOM detection on an image using OCR.
    
    Args:
        image: PIL Image
        ocr_reader: Optional pre-initialized EasyOCR reader
    
    Returns:
        annotated_image: PIL Image with numbered marks
        bboxes: List of normalized bboxes (y, x, h, w)
        texts: List of detected text strings
    """
    if ocr_reader is None:
        ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available(), verbose=False)
    
    width, height = image.size
    image_np = np.array(image)
    
    # Detect text regions
    ocr_results = ocr_reader.readtext(image_np, text_threshold=0.5)
    
    bboxes = []
    texts = []
    for detection in ocr_results:
        coords, text, confidence = detection
        x1, y1 = coords[0]
        x2, y2 = coords[2]
        y_norm = y1 / height
        x_norm = x1 / width
        h_norm = (y2 - y1) / height
        w_norm = (x2 - x1) / width
        bboxes.append((y_norm, x_norm, h_norm, w_norm))
        texts.append(text)
    
    # Annotate image with marks
    if len(bboxes) > 0 and som_available:
        mark_helper = MarkHelper()
        annotated = plot_boxes_with_marks(
            image.copy(), bboxes, mark_helper,
            edgecolor=(255, 0, 0), linewidth=2,
            normalized_to_pixel=True, add_mark=True
        )
    elif len(bboxes) > 0:
        annotated = image.copy()
        draw = ImageDraw.Draw(annotated)
        for i, (y_n, x_n, h_n, w_n) in enumerate(bboxes):
            x1 = int(x_n * width)
            y1 = int(y_n * height)
            x2 = int((x_n + w_n) * width)
            y2 = int((y_n + h_n) * height)
            draw.rectangle([x1, y1, x2, y2], outline='red', width=2)
            draw.text((x1, y1-15), str(i), fill='blue')
    else:
        annotated = image.copy()
    
    return annotated, bboxes, texts


# Initialize OCR reader (reuse across calls)
ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available(), verbose=False)
print(f"OCR reader initialized (GPU: {torch.cuda.is_available()})")

In [ ]:
# Load Magma model
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_DTYPE = torch.bfloat16
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_MODEL = "microsoft/Magma-8B"

print(f"Loading Magma model on {DEVICE}...")
magma_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, trust_remote_code=True,
    torch_dtype=MODEL_DTYPE, device_map="auto",
    attn_implementation="eager"
)
magma_processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
magma_model.eval()
print("Model loaded")

In [ ]:
import re
import json

def ask_magma_som(image, task_description, previous_actions=""):
    """Query Magma with a SOM-annotated image for UI navigation.
    
    Returns raw response string.
    """
    prompt = f'''<image>
Imagine that you are imitating humans doing web navigation for a task step by step. At each stage, you can see the webpage like humans by a screenshot and know the previous actions before the current step decided by yourself through recorded history. You need to decide on the following action to take. You can click an element with the mouse, select an option, or type text with the keyboard. The output format should be a dictionary like: 
"{{"ACTION": "CLICK" or "TYPE" or "SELECT", "MARK": a numeric id, e.g., 5, "VALUE": a string value for the action if applicable, otherwise None}}".
You are asked to complete the following task: {task_description}. The previous actions you have taken: 
{previous_actions if previous_actions else "(None)"}
For your convenience, I have labeled the candidates with numeric marks and bounding boxes on the screenshot. What is the next action you would take?
'''
    if hasattr(magma_model.config, 'mm_use_image_start_end') and magma_model.config.mm_use_image_start_end:
        prompt = prompt.replace('<image>', '<image_start><image><image_end>')
    
    convs = [
        {"role": "system", "content": "You are agent that can see, talk and act."},
        {"role": "user", "content": prompt},
    ]
    formatted_prompt = magma_processor.tokenizer.apply_chat_template(
        convs, tokenize=False, add_generation_prompt=True
    )
    
    inputs = magma_processor(images=[image], texts=formatted_prompt, return_tensors="pt")
    inputs['pixel_values'] = inputs['pixel_values'].unsqueeze(0)
    inputs['image_sizes'] = inputs['image_sizes'].unsqueeze(0)
    inputs = inputs.to(MODEL_DTYPE).to(DEVICE)
    
    magma_model.generation_config.pad_token_id = magma_processor.tokenizer.pad_token_id
    with torch.inference_mode():
        output_ids = magma_model.generate(
            **inputs, temperature=0.0, do_sample=False,
            num_beams=1, max_new_tokens=256, use_cache=False
        )
    
    prompt_decoded = magma_processor.batch_decode(inputs['input_ids'], skip_special_tokens=True)[0]
    response = magma_processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    return response.replace(prompt_decoded, '').strip()


def parse_magma_action(response):
    """Parse Magma response into {ACTION, MARK, VALUE}."""
    result = {"ACTION": None, "MARK": None, "VALUE": None, "raw": response, "parse_error": False}
    try:
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            result["ACTION"] = parsed.get("ACTION")
            result["MARK"] = parsed.get("MARK")
            result["VALUE"] = parsed.get("VALUE")
    except json.JSONDecodeError:
        result["parse_error"] = True
    return result

In [ ]:
# Test on a single image
image_path = "path/to/your/image.png"  # <-- Change this
task = "Click the search button"        # <-- Change this

image = Image.open(image_path)
som_image, bboxes, texts = process_som(image, ocr_reader)

print(f"Detected {len(bboxes)} elements:")
for i, t in enumerate(texts):
    print(f"  [{i}] {t}")
display(som_image)

# Run inference
response = ask_magma_som(som_image, task)
parsed = parse_magma_action(response)
print(f"\nResponse: {response}")
print(f"ACTION: {parsed['ACTION']}, MARK: {parsed['MARK']}, VALUE: {parsed['VALUE']}")

In [ ]:
def preprocess_dataset(dataset, ocr_reader, output_dir="som_processed"):
    """Batch process a dataset: run SOM on each image and save results.
    
    Args:
        dataset: Iterable of dicts with 'image' (PIL) and 'action' fields
        ocr_reader: EasyOCR reader instance
        output_dir: Directory to save annotated images and metadata
    
    Returns:
        List of processed samples with SOM annotations
    """
    os.makedirs(output_dir, exist_ok=True)
    results = []
    
    for idx, sample in enumerate(dataset):
        image = sample['image'] if isinstance(sample['image'], Image.Image) else Image.open(sample['image'])
        som_image, bboxes, texts = process_som(image, ocr_reader)
        
        # Save annotated image
        som_path = os.path.join(output_dir, f"som_{idx:06d}.png")
        som_image.save(som_path)
        
        results.append({
            "index": idx,
            "som_image_path": som_path,
            "bboxes": bboxes,
            "texts": texts,
            "num_elements": len(bboxes),
        })
        
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1} images...")
    
    # Save metadata
    metadata_path = os.path.join(output_dir, "metadata.json")
    with open(metadata_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Done. {len(results)} images processed. Metadata saved to {metadata_path}")
    
    return results